In [4]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv(override=True)

api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise ValueError("API Key not found in .env")

client = genai.Client(api_key=api_key)


In [5]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm_gemini = ChatGoogleGenerativeAI(model="gemini-3.8-flash", temperature=0.7)

result = llm_gemini.invoke(
    "Tell me a joke. Generate the output in key-value pair format with the following keys: setup, punchline"
)

print(result.content)

[{'type': 'text', 'text': '```json\n{\n  "setup": "Why do programmers prefer dark mode?",\n  "punchline": "Because light attracts bugs."\n}\n```', 'extras': {'signature': 'ErQLCrELAWkUfRP7cjSq1sruR51g88MBX4iyem+HcfBPrVHbCVxW/AGUsrV4+wObUboctssQGKsWKXpBb+o1VmW2mT2L+h+LNExA/Ve1Tqgr84wOE90uJAMiEXxjasYXQJrXKSAkuJHeao8K6FbrS+K9z2rKMpG0nAuAni7qMNsmlTNFW7r/6FHJnq2DG6jEYHmiyWPqvrTmGqnYuF33d039Viyy/Yr7hyLxfiAWIwi9bxCufsenIr4QZQWyZjasC90oiSYWjtyeFgcQyDw6PXPpsI10Ues9FFYTV2IdP6jI63g4gjkbxuKIJONP3zObhHVyEwYBheyB06KDRk2Su/4XjxC3OVBNbG5d8RlOwp5ZfImsgGS1ATK26UwBOrvADgGlH4ThBcAmjIj+p6lBJ7T+6lsK4Is8Gg3HBD6mJY7YBQYwIa0Qj3LpqS27gePG+VkthZiNTPW3T0Dedy28nk2KuAvVLQQtWbyEP5D85CLcVpdvoTbwvJdSiS3/4l2iPc158C3HDbp4UxHRsdaBLtDm9TkKW6prv1ox51fXLNuSj1e9Q/EPXA0/C/IEwu2EM8gLupPM3qkvbU+S8kx3WQ019CP2ShUHPWgPGU4sWM+sI51pmqVMZGi0MLN6i6AOOd8u5Z1OGyOckdHnmkO8Girabcu9KmNF8MRsI1C9t2YX3D8hl+huLNd0qdT73Uq4EecsQMf84Ku3oQIZU4KNzP4k/dFrELbVvFf6m0aYStlR7BPt2BI8NGzdKNA4jpdy80fMpT8/jNpcmE3kVMaOBVKKh8UT3t4caNPjFFQv/ILOYEsA2lc1pLe8Wnvr

In [1]:
from pydantic import BaseModel

class LLMSchema(BaseModel):
    setup: str
    punch_line: str

In [2]:
obj = LLMSchema(
    setup="Why did the scarecrow win an award?",
    punch_line="Because he was outstanding in his field."
)

print(obj)

setup='Why did the scarecrow win an award?' punch_line='Because he was outstanding in his field.'


In [ ]:
# from pydantic import BaseModel, ConfigDict

# class LLMSchema(BaseModel):
#     model_config = ConfigDict(extra="forbid")

#     setup: str
#     punch_line: str

In [6]:
from pydantic import BaseModel, Field

class LLMSchema(BaseModel):
    setup: str = Field(
        description="The setup or opening question of the joke."
    )

    punch_line: str = Field(
        description="The punch line or funny ending of the joke.")

In [8]:
structured_llm = llm_gemini.with_structured_output(LLMSchema)

In [9]:
result = structured_llm.invoke("Tell me a joke")

print(result)
print(type(result))

setup='Why do programmers prefer dark mode?' punch_line='Because light attracts bugs.'
<class '__main__.LLMSchema'>


In [10]:
result.model_dump()

{'setup': 'Why do programmers prefer dark mode?',
 'punch_line': 'Because light attracts bugs.'}

In [11]:
from typing import TypedDict

class LLMSchemaTD(TypedDict):
    setup: str
    punch_line: str

In [ ]:
# obj: LLMSchemaTD = {
#     "setup": "Some setup",
#     "punchline": "Some punch line"
# }

In [13]:
from typing import TypedDict

class LLMSchemaTD(TypedDict):
    setup: str
    punch_line: str

structured_llm = llm_gemini.with_structured_output(LLMSchemaTD)

result = structured_llm.invoke("Tell me a joke")

print(result)
print(type(result))

{'setup': 'Why do programmers prefer dark mode?', 'punch_line': 'Because light attracts bugs.'}
<class 'dict'>


In [14]:
print(result["setup"])
print(result["punch_line"])

Why do programmers prefer dark mode?
Because light attracts bugs.


In [17]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template =  ChatPromptTemplate.from_messages([

    ("system", "You are a helpful assistant."),
    ("human", "{input}")
])

In [18]:
response = llm_gemini.invoke("tell me fact about football")
print(response.content)

[{'type': 'text', 'text': "Here is one of the most bizarre and incredible facts about football (soccer):\n\n**The highest-scoring professional football match in history ended 149–0.**\n\nOn October 31, 2002, two top clubs in Madagascar—**AS Adema** and **SO l'Emyrne (SOE)**—faced each other. In the previous match, SOE had been eliminated from the title race by a controversial, late penalty kick awarded by the referee. \n\nTo protest the officiating, as soon as the whistle blew, the SOE players turned around and deliberately kicked the ball into their own net. Every time Adema kicked off to restart the match, SOE took the ball back and scored another own goal. \n\nBy the end of the 90 minutes, **SOE had scored 149 own goals**. The opposing team, AS Adema, literally did nothing the entire game except watch, but they still won 149–0, setting a Guinness World Record.\n\n***\n\n**A few quick bonus facts:**\n* **Ancient Origins:** While modern football was codified in England in 1863, the ea

In [24]:
from langchain_core.output_parsers import StrOutputParser

string_parser = StrOutputParser()


In [19]:
user_input = {
    "input": "What is the capital of France?"}

In [20]:
ready_prompt = prompt_template.invoke(user_input)

In [22]:
response = llm_gemini.invoke(ready_prompt)

In [25]:
final_result = string_parser.invoke(response)

print(final_result)

The capital of France is **Paris**.
